# Homework 2: Probability Models

BEE 4850/5850

**Name**: Christine Swanson 

**ID**: cms549

> **Due Date**
>
> Friday, 2/21/25, 9:00pm

## Overview

### Instructions

The goal of this homework assignment is to practice developing and
working with probability models for data.

-   Problem 1 asks you to fit a sea-level rise model using normal
    residuals and to assess the validity of that assumption.
-   Problem 2 asks you to model the time series of hourly
    weather-related variability at a tide gauge using an autoregressive
    model.
-   Problem 3 asks you to use Poisson regression to predict salamander
    counts based on environmental data.
-   Problem 4 (only required for students in BEE 5850) asks you to look
    at the impact of the gender of hurricane names on deaths[1].

[1] Yes, seriously. Ish. Trust me, I know.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [ ]:
# commented out - using Python 
#import Pkg
#Pkg.activate(@__DIR__)
#Pkg.instantiate()

The following packages are included in the environment (to help you find
other similar packages in other languages). The code below loads these
packages for use in the subsequent notebook (the desired functionality
for each package is commented next to the package).

In [6]:
# make sure to reference ChatGPT for equivalent Python libraries 
import numpy as np  # random number generation and seed-setting
import pandas as pd  # tabular data structure (equivalent to DataFrames)
import csv  # read/write .csv files
import scipy.stats as stats  # interface to work with probability distributions
import matplotlib.pyplot as plt  # plotting library (equivalent to Plots in Julia)
import seaborn as sns  # additional statistical plotting tools (equivalent to StatsPlots in Julia)
from scipy.optimize import minimize  # optimization tools (equivalent to Optim in Julia)
from scipy.stats import norm  # normal distribution

## Problems

### Scoring

-   Problem 1 is worth 7 points.
-   Problem 2 is worth 6 points.
-   Problem 3 is worth 7 points.
-   Problem 4 is worth 5 points.

### Problem 1

Consider the following sea-level rise model from [Grinsted et al
(2010)](https://doi.org/10.1007/s00382-008-0507-2), which models
sea-level rise based on a linear relationship between global mean
temperature and an “equilibrium” sea level:

$$\begin{align*}
\frac{dS}{dt} &= \frac{S_\text{eq} - S}{\tau} \\
S_\text{eq} &= aT + b,
\end{align*}
$$

where

-   $S(t)$ is the global mean sea level (in mm) at time $t$;
-   $\tau$ is the response time of sea level (in yrs);
-   $S_\text{eq}$ is the equilibrium sea-level (in mm) at temperature
    $T$ (in $^\circ$C);
-   $a$ is the sensitivity of $S_\text{eq}$ to $T$ (in mm/$^\circ$C);
-   $b$ is the intercept of $S_\text{eq}$, or the $S_\text{eq}$ when
    $T=0^\circ$C (in mm).

**In this problem**:

-   Load the data from the `data/` folder and, following Grinsted et al
    (2010), normalize both datasets to the 1980-1999 mean (subtract that
    mean from the data).
    -   Global mean temperature data from the HadCRUT 5.0.2.0 dataset
        (<https://hadobs.metoffice.gov.uk/hadcrut5/data/HadCRUT.5.0.2.0/download.html>)
        can be found in
        `data/HadCRUT.5.0.2.0.analysis.summary_series.global.annual.csv`.
        This data is averaged over the Northern and Southern Hemispheres
        and over the whole year.
    -   Global mean sea level anomalies (relative to the 1990 mean
        global sea level) are in `data/CSIRO_Recons_gmsl_yr_2015.csv`,
        courtesy of CSIRO
        (<https://www.cmar.csiro.au/sealevel/sl_data_cmar.html>). The
        standard deviation of the estimate is also added for each year.
-   Write a function to simulate global mean sea levels under a set of
    model parameters after discretizing the equations above with a
    timestep of $\delta t = 1$ yr. You will need to subset the
    temperature data to the years where you also have sea-level data.
-   Fit the model under the assumption of Gaussian i.i.d. residuals
    (include both an uncertain model error term and the standard
    deviation of the observations in the probability model
    specification) by maximizing the likelihood. Report the parameter
    estimates. Note that you will need another parameter $S_0$ for the
    initial sea level. What can you conclude about the relationship
    between global mean temperature increases and global mean sea level
    rise rates?
-   How appropriate was the Gaussian i.i.d. probability model for the
    residuals? Use any needed quantitative or qualitative assessments of
    goodness of fit to justify your answer. If this was not an
    appropriate probability model, what would you change?

In [13]:
# load the data

# global mean temp data
temp_data = pd.read_csv("./data/HadCRUT.5.0.2.0.analysis.summary_series.global.annual.csv", sep=",", header=0)

# sea level data
sea_lvl_data = pd.read_csv("./data/CSIRO_Recons_gmsl_yr_2015.csv", sep=",", header=0)

In [14]:
# normalize the data by subtracting off the mean from 1980-1999
temp_data["Anomaly_deg_C_normalized"] = temp_data["Anomaly (deg C)"] - np.mean(temp_data["Anomaly (deg C)"][130:150]) # 1980-1999 indices
sea_lvl_data["GMSL_mm_normalized"] = sea_lvl_data["GMSL (mm)"] - np.mean(sea_lvl_data["GMSL (mm)"][130:150]) # 1980-1999 indices

In [ ]:
# subset the temp data to years we have sea level data (1880-2013)
temp_data = temp_data[(temp_data["Time"] >= 1880) & (temp_data["Time"] <= 2013)]

temp_data = temp_data.reset_index(drop=True) # need to reset the index after filtering the rows 

In [28]:
# function to simulate global mean sea levels under set of model params after discretizing eqns 

# create model for sea level based on temp
def sea_level_model(global_temp, p=(129, 770, 208, -2)): # params = a, b, tau, S0 from Grinsted et al. (2010)

    # define our unknown param values based on input to function
    a, b, tau, S0 = p

    dt = 1 # 1 year
    
    T = global_temp

    S = np.zeros(len(T))

    S[0] = S0

    for i in range(len(T) - 1):
        S[i+1] = S[i] + ((a * T[i] + b - S[i])/tau)*dt
    return S

def sea_level_model_wrap(params):
    return sea_level_model(global_temp=temp_data['Anomaly_deg_C_normalized'], p=params)

In [29]:
# fit model under assumption of Gaussian iid residuals by MLE

def gaussian_iid_homosked(params, sea_level_data, sea_level_error, m):
    # note - sigma here is standard deviation of the discrepancy term
    a, b, tau, S0, sigma = params
    sea_level_sim = m((a, b, tau, S0))

    # documentation scipy: loc is mean, scale is standard deviation
    ll = np.sum(norm.logpdf(sea_level_data, loc=sea_level_sim, scale=np.sqrt(sigma**2 + sea_level_error**2)))
    return ll

# a, b, tau, S0, sigma (note, Grinsted et al. units are in m, our units are in mm)
lower = [100, 700, 208, -2, 1.0]
upper = [2000, 1500, 300, 550, 10.0]

p0 = [129, 770, 208, -2, 1.0]

# result = minimize(lambda params: -gaussian_iid_homosked(params, sea_level_data = sea_filt['GMSL_anom_normalized'], sea_level_error=sea_filt['GMSL uncertainty (mm)'], m=sea_level_model_wrap), p0, bounds=list(zip(lower, upper)))
# θ_iid = result.x
def neg_log_likelihood(params):
    return -gaussian_iid_homosked(
        params,
        sea_level_data=sea_lvl_data["GMSL_mm_normalized"],
        sea_level_error=sea_lvl_data["GMSL uncertainty (mm)"],
        m=sea_level_model_wrap
    )

result = minimize(neg_log_likelihood, p0, bounds=list(zip(lower, upper)))

θ_iid = result.x

In [ ]:
# print the parameter estimates
print("Estimate of 'a': ", θ_iid[0], "mm/deg C")
print("Estimate of 'b': ", θ_iid[1], "mm")
print("Estimate of tau: ", θ_iid[2], "years")
print("Estimate of S0: ", θ_iid[3], "mm")
print("Estimate of sigma: ", θ_iid[4], "mm") # not sure if these are the correct units for std dev of discrepancy term

Estimate of 'a':  2000.0 mm/deg C
Estimate of 'b':  700.0 mm
Estimate of tau:  208.0 years
Estimate of S0:  -2.0 mm
Estimate of sigma:  10.0 mm


**RESPONSE:**

### Problem 2

Tide gauge data is complicated to analyze because it is influenced by
different harmonic processes (such as the linear cycle). In this
problem, we will develop a model for this data using [NOAA data from the
Sewell’s Point tide
gauge](https://tidesandcurrents.noaa.gov/waterlevels.html?id=8638610)
outside of Norfolk, VA from `data/norfolk-hourly-surge-2015.csv`. This
is hourly data (in m) from 2015 and includes both the observed data
(`Verified (m)`) and the tide level predicted by NOAA’s sinusoidal model
for periodic variability, such as tides and other seasonal cycles
(`Predicted (m)`).

**In this problem**:

-   Load the data file. Take the difference between the observations and
    the sinusoidal predictions to obtain the tide level which could be
    attributed to weather-related variability (since for one year
    sea-level rise and other factors are unlikely to matter). Plot this
    data.
-   Develop an autoregressive (AR) model for the weather-related
    variability in the Norfolk tide gauge. Make sure to include your
    logic or exploratory analysis used in determining the model
    specification.
-   Use your model to simulate 1,000 realizations of hourly tide gauge
    observations by adding simulations from your AR model back to the
    predicted sinusoidal trend. What is the distribution of the maximum
    tide level? How does this compare to the observed value?

## Problem 3

The file `data/salamanders.csv` contains counts of salamanders from 47
different plots of the same area in California, as well as the
percentage of ground cover and age of the forest in the plot. You would
like to see if you can use these data to predict the salamander counts
with a Poisson regression.

**In this problem**:

-   Load the data. You may need to standardize the predictors as they
    are much larger than the counts.
-   Fit a Poisson regression model for salamander counts using the
    percentage of ground cover.
-   Plot the expected counts and 90% prediction intervals from your
    model. How well does the model predict the observed counts? In what
    ways does it do a good or bad job?
-   Can you improve the model by including forest age? Why do you think
    this helps or does not help with prediction?

### Problem 4

<span style="color:red;">GRADED FOR 5850 STUDENTS ONLY</span>

In 2014, [a paper was published in a prestigious
journal](https://www.pnas.org/doi/10.1073/pnas.1402786111) which claimed
that hurricanes with more feminine names are deadlier than hurricanes
with more masculine names because people take warnings about
female-named hurricanes less seriously[1]. The file
`data/Hurricanes.csv` contains the original data used in this analysis.
While we won’t replicate the specific analysis in this paper, let’s use
the data to look at this hypothesis.

**In this problem**:

-   One might interpret the hypothesis to claim that the impact of the
    name is strengthened by the the more powerful. A measure of
    hurricane strength is its minimum pressure (`min_pressure` in the
    dataset). Fit a model that predicts deaths (‘deaths’) using the
    femininity of the name (`femininity`) and minimum pressure (you may
    need to standardize the pressure).
-   Interpret the results by generating counterfactual simulations for
    hurricanes with the most feminine and masculine name scores. Plot
    the expected values and 90% prediction intervals from these two sets
    of simulations and compare with the observed storm deaths. Where
    does the model do well or not well? Does the effect size of the
    gender of the name seem plausible?
-   Conclude with a summary of your conclusions about the impact of the
    gender of a hurricane’s name on deaths. How might you change the
    approach in this problem to keep exploring this hypothesis, if at
    all?

[1] This paper has become a bit of a joke among statisticians, but let’s
take the hypothesis seriously for this problem’s sake.

## References